# M13 prerequisite — the FID floor at small sample counts

M11 scored every sampler with FID on **5,000** samples. Even the converged run
(DPM-Solver-3, 99 NFE) came out at **16.5**, while the paper reports ~4–5 for this
checkpoint with 50,000 samples. How much of that gap is simply the small sample
count? FID is biased upward when the sample is small, even for perfect samples.

This notebook measures that bias directly. It scores **real, held-out CIFAR-10
test images** — perfect "samples" — with *exactly* M11's FID call (clean-fid,
CIFAR-10 training statistics, `mode="legacy_tensorflow"`), at 1k, 2k, 5k and 10k
images, with several random draws at each size.

- The 5k value is the **floor**: the best FID any sampler could get in M11's setup.
- `converged FID − floor` is the gap that is genuinely about the samples.
- The 10k value is the classic "CIFAR-10 test vs train" FID; clean-fid documents it
  as ≈ 3.1 in this mode, so it also checks the pipeline itself.

No diffusion model is involved. On a Kaggle T4 this takes ~2–3 minutes, mostly
downloads.

### How to run this on Kaggle

1. Push first (the notebook clones `origin/main`).
2. New Notebook → **Accelerator: GPU T4 ×1**, **Internet: ON** → Run All.
3. Download `results/tier3_fid_floor.csv`, `figures/11b_fid_floor.png` and this
   notebook (with outputs) from `/kaggle/working/NUMERICAL-PROJECT/`, commit.

Local smoke test (tiny sizes, temp-dir outputs):
`M11B_SMOKE=1 jupyter nbconvert --to notebook --execute notebooks/11b_fid_floor.ipynb --output /tmp/nb11b.ipynb`

In [ ]:
import os, sys, pathlib, subprocess, time, tempfile, io, socket

IN_KAGGLE = pathlib.Path("/kaggle").exists()
SMOKE = os.environ.get("M11B_SMOKE") == "1"
REPO = "https://github.com/MehemudAzad/NUMERICAL-PROJECT.git"

if IN_KAGGLE:
    ROOT = pathlib.Path("/kaggle/working/NUMERICAL-PROJECT")
    if ROOT.exists():
        subprocess.run(["git", "fetch", "--depth", "1", "origin", "main"], cwd=ROOT, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO], cwd="/kaggle/working", check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "clean-fid", "pyarrow"], check=True)
else:
    p = pathlib.Path.cwd()
    while not (p / "pyproject.toml").exists() and p != p.parent:
        p = p.parent
    ROOT = p

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from cleanfid import fid as cleanfid

# A stalled download must fail, not hang (urllib otherwise waits forever).
socket.setdefaulttimeout(120)

# clean-fid calls scipy.linalg.sqrtm(..., disp=False), an argument SciPy 1.18 removed
# (the laptop's pinned version; Kaggle's older SciPy still has it, which is why
# notebook 11 ran there unchanged). Restore the old call signature only when it is
# missing, so on Kaggle this is a no-op and the FID is computed exactly as in M11.
import inspect
import scipy.linalg as _sl
if "disp" not in inspect.signature(_sl.sqrtm).parameters:
    _sqrtm_new = _sl.sqrtm

    def _sqrtm_compat(A, disp=True, blocksize=64):
        X = _sqrtm_new(A)
        if disp:
            return X
        # the old API's second return value: relative Frobenius residual
        return X, np.linalg.norm(X @ X - A, "fro") ** 2 / np.linalg.norm(A, "fro")

    _sl.sqrtm = _sqrtm_compat
    print("scipy.linalg.sqrtm: compatibility shim active (SciPy without `disp`)")

if SMOKE:
    OUT = pathlib.Path(tempfile.mkdtemp(prefix="m11b_smoke_"))
    RESULTS, FIGURES = OUT / "results", OUT / "figures"
else:
    RESULTS, FIGURES = ROOT / "results", ROOT / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SIZES = [100, 200] if SMOKE else [1000, 2000, 5000, 10000]
DRAWS = 2 if SMOKE else 3            # random subsets per size (10k is the whole test set: 1 draw)
print(f"root {ROOT}\ndevice {DEVICE}\nsizes {SIZES}, {DRAWS} draws each"
      + ("\nSMOKE: outputs in a temp dir, not the repo" if SMOKE else ""))

## 1. The real images

The 10,000 CIFAR-10 **test** images: real data the training statistics never saw.
They are fetched from Hugging Face (24 MB); torchvision's Toronto download (170 MB)
is only a fallback.

In [ ]:
def load_cifar10_test():
    # (10000, 32, 32, 3) uint8, plus where it came from.
    try:
        from huggingface_hub import hf_hub_download
        import pyarrow.parquet as pq
        path = hf_hub_download("uoft-cs/cifar10", "plain_text/test-00000-of-00001.parquet",
                               repo_type="dataset")
        col = pq.read_table(path, columns=["img"]).column("img").to_pylist()
        arr = np.stack([np.asarray(Image.open(io.BytesIO(d["bytes"])).convert("RGB")) for d in col])
        return arr, "Hugging Face uoft-cs/cifar10, test split"
    except Exception as e:
        print(f"Hugging Face route failed ({type(e).__name__}: {e}); using torchvision")
        import torchvision
        ds = torchvision.datasets.CIFAR10(root=str(pathlib.Path(tempfile.gettempdir()) / "cifar10"),
                                          train=False, download=True)
        return ds.data, "torchvision CIFAR10, test split"


t0 = time.time()
test_imgs, source = load_cifar10_test()
print(f"{len(test_imgs)} test images from {source} ({time.time()-t0:.1f}s)")
assert test_imgs.shape[1:] == (32, 32, 3) and test_imgs.dtype == np.uint8

## 2. FID of real images, with M11's exact FID call

For each size *N* and each draw, a random subset of *N* test images is written as
PNGs — the same file format M11's samples were scored from — and passed to the same
`cleanfid.compute_fid(...)` call.

In [ ]:
def fid_of(images_uint8):
    # Score a (N, 32, 32, 3) uint8 array exactly as notebook 11 scored its samples.
    with tempfile.TemporaryDirectory() as tmp:
        for k, im in enumerate(images_uint8):
            Image.fromarray(im).save(pathlib.Path(tmp) / f"{k:06d}.png")
        return cleanfid.compute_fid(tmp, dataset_name="cifar10", dataset_res=32,
                                    dataset_split="train", mode="legacy_tensorflow",
                                    device=DEVICE, verbose=False,
                                    # single-process loading: clean-fid's worker processes
                                    # cannot pickle its resize lambda on macOS/Windows (spawn).
                                    # Only affects loading speed, never the score.
                                    num_workers=0)


rows = []
t0 = time.time()
for n in SIZES:
    n = min(n, len(test_imgs))
    draws = 1 if n == len(test_imgs) else DRAWS
    for seed in range(draws):
        idx = np.random.default_rng(seed).choice(len(test_imgs), n, replace=False)
        score = fid_of(test_imgs[idx])
        rows.append(dict(n_samples=n, seed=seed, fid=score, source=source,
                         method="clean-fid CIFAR-10 train stats, legacy_tensorflow"))
        print(f"  N={n:6d} draw {seed}: FID {score:6.2f}   [{time.time()-t0:6.1f}s]")

floor = pd.DataFrame(rows)
floor.to_csv(RESULTS / "tier3_fid_floor.csv", index=False)
summary = floor.groupby("n_samples").fid.agg(["mean", "std", "count"]).round(3)
summary

## 3. What the floor says about M11's numbers

In [ ]:
m11 = pd.read_csv(ROOT / "results" / "tier3_fid.csv")
conv = float(m11.loc[m11.config == "dpm3_converged", "fid"].iloc[0])
n_m11 = int(m11.n_samples.iloc[0])
by_n = floor.groupby("n_samples").fid.mean()
floor_m11 = by_n.get(n_m11, np.interp(n_m11, by_n.index, by_n.values))

print(f"M11 sample size                : {n_m11}")
print(f"FID floor at that size (real)  : {floor_m11:.2f}   <- best any sampler could score in M11")
print(f"M11 converged DPM-3 (99 NFE)   : {conv:.2f}")
print(f"gap that is about the samples  : {conv - floor_m11:.2f}")
print(f"real test vs train, largest N  : {by_n.iloc[-1]:.2f}  (clean-fid reports ~3.1 at 10k)")
print("\nM11 at 10 NFE, with the floor subtracted:")
at10 = m11[m11.nfe.isin([9, 10])].copy()
at10["fid_minus_floor"] = at10.fid - floor_m11
print(at10[["config", "nfe", "fid", "fid_minus_floor"]].round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.errorbar(summary.index, summary["mean"], yerr=summary["std"].fillna(0), fmt="o-", capsize=3,
            label="real CIFAR-10 test images (the floor)")
ax.axhline(conv, color="tab:red", ls="--", lw=1.2, label=f"M11 converged DPM-3, N={n_m11}")
ax.axvline(n_m11, color="0.5", ls=":", lw=1)
ax.set_xscale("log")
ax.set_xlabel("number of images scored")
ax.set_ylabel("FID vs CIFAR-10 train statistics")
ax.set_title("How much FID comes from the sample size alone")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "11b_fid_floor.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for f in [RESULTS / "tier3_fid_floor.csv", FIGURES / "11b_fid_floor.png"]:
    assert f.exists(), f"missing deliverable: {f}"
    print(f"  ok  {f}")
print("\nFID floor: DONE" + ("   (smoke run -- sizes too small to mean anything)" if SMOKE else ""))
if IN_KAGGLE:
    print("Download results/tier3_fid_floor.csv, figures/11b_fid_floor.png and this notebook, then commit.")